In [23]:
import pandas as pd
from string import Template
from pathlib import Path

DATA_FILE = Path("../data/sinhala.xlsx")
TEMPLATE_FILE = Path("../gemma_zeroshot_template.txt")
OUTPUT_FILE_TRAIN = Path("data/lanka/sinhala/train.csv")
OUTPUT_FILE_TEST = Path("data/lanka/sinhala/test.csv")


In [24]:
# --- read files ---
if not DATA_FILE.exists():
    raise FileNotFoundError("syllabus.csv not found.")

if not TEMPLATE_FILE.exists():
    raise FileNotFoundError("template not found.")

df = pd.read_excel(DATA_FILE)
template_text = TEMPLATE_FILE.read_text(encoding="utf-8")
tmpl = Template(template_text)

# --- fill prompts ---
filled = []
for idx, row in df.iterrows():
    mapping = {}
    for k, v in row.items():
        if pd.isna(v):
            mapping[k] = ""
        else:
            mapping[k] = str(v)

    try:
        out = tmpl.substitute(mapping)   # strict mode → fail on missing key
    except KeyError as e:
        raise KeyError(f"Missing placeholder value for: {e.args[0]}") from None

    filled.append(out)

# --- write result ---
df["prompt"] = filled


In [26]:
# Columns that should stay the same
id_cols = ['country', 'curriculum', 'addressing', 'topic_list', 'language',
       'grade', 'topic', 'prompt']

# Example columns
example_cols = [f"example_{i}" for i in range(1, 11)]

df_long = df.melt(
    id_vars=id_cols,
    value_vars=example_cols,
    var_name="example_num",
    value_name="output"
)

# Optional: remove rows where an example is missing
df_long = df_long.dropna(subset=["output"])

# Optional: drop the example_num column if you don't need it
df_long = df_long.drop(columns=["example_num"])



In [29]:
df = df[['language', 'grade', 'topic', 'prompt']]

df.to_csv(OUTPUT_FILE_TEST, index=False)

print("Done. Wrote:", OUTPUT_FILE_TEST)

Done. Wrote: data/lanka/sinhala/test.csv


In [27]:
df_long = df_long[["prompt", "output"]]
df_long.to_csv(OUTPUT_FILE_TRAIN, index=False)

print("Done. Wrote:", OUTPUT_FILE_TRAIN)

Done. Wrote: data/lanka/sinhala/train.csv
